In [1]:
import os
import numpy as np

import jax
import jax.numpy as jnp

import functools


/home/ron/scratch/.venv/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [2]:
# Question 1.1

mesh = jax.make_mesh((4, 2), ('X','Y'))

average_shmap = jax.shard_map(
    lambda x: x.mean(keepdims=True),
    mesh=mesh,
    in_specs=jax.P('X','Y'), out_specs=jax.P('X','Y')
)

@functools.partial(jax.jit, out_shardings=jax.NamedSharding(mesh, jax.P('X','Y')))
def average_jit(x):
  X, Y = mesh.axis_sizes
  x = jax.lax.reshape(
      x,
      (X, x.shape[0] // X, Y, x.shape[1] // Y),
      out_sharding=jax.NamedSharding(mesh, jax.P('X', None, 'Y', None)),
  )

  return x.mean(axis=(1, 3))

x = jnp.arange(8 * 64 * 8, dtype=jnp.float32).reshape(8 * 64, 8)
x = jax.device_put(x, jax.NamedSharding(mesh, jax.P('X','Y')))

y1 = average_shmap(x)
y2 = average_jit(x)

np.testing.assert_array_equal(y1, y2)

In [3]:
# Question 1.2

mesh = jax.make_mesh((4, 2), ('X','Y'))

def shift_shmap(x, shift: int):
  shmapped = jax.shard_map(
      lambda x: jnp.roll(x, shift, axis=0),
      mesh=mesh,
      in_specs=jax.P('X','Y'), out_specs=jax.P('X','Y')
  )
  return shmapped(x)

@functools.partial(jax.jit, static_argnames=['shift'], out_shardings=jax.NamedSharding(mesh, jax.P('X','Y')))
def shift_jit(x, shift: int):
  X, Y = mesh.axis_sizes
  reshaped = x.reshape(X, x.shape[0] // X, -1)
  return jnp.roll(reshaped, shift, axis=1).reshape(x.shape[0], x.shape[1])

x = jnp.arange(8 * 64 * 8, dtype=jnp.float32).reshape(8 * 64, 8)
x = jax.device_put(x, jax.NamedSharding(mesh, jax.P('X','Y')))

y1 = shift_shmap(x, 5)
y2 = shift_jit(x, 5)

np.testing.assert_array_equal(y1, y2)

In [4]:

# Question 2.1


def moe_local(W: jnp.ndarray, A: jnp.ndarray, B: jnp.ndarray) -> jnp.ndarray:
    S, _ = A.shape
    E, _, F = W.shape

    output = jnp.zeros((S, F), dtype=A.dtype)
    for e in range(E):
        mask = B == e
        selected = jnp.where(mask[:, None], A, 0.0)
        output = output + selected @ W[e]

    return output


def moe_reference(W, A, B):
    return jax.vmap(lambda a, b: a @ W[b])(A, B)


# Keep the single-device baseline small enough to fit on one TPU chip.
E = 8
S, D, F = 1024, 512, 512

# Use a larger problem for the sharded experiments. 2.1 does not use these.
# If this OOMs, reduce S_big first; if it compiles too slowly, reduce F_big.
S_big, D_big, F_big = 8192, 1024, 2048

key = jax.random.key(0)
k1, k2, k3 = jax.random.split(key, 3)

W0 = jax.random.normal(k1, (E, D, F))
A0 = jax.random.normal(k2, (S, D))
B0 = jax.random.randint(k3, (S,), 0, E, dtype=jnp.int32)


np.testing.assert_allclose(
    moe_local(W0, A0, B0),
    moe_reference(W0, A0, B0),
    rtol=1e-1,
    atol=1e-1,
)

print("2.1 local config:  ", (E, S, D, F))
print("sharded config:   ", (E, S_big, D_big, F_big))


2.1 local config:   (8, 1024, 512, 512)
sharded config:    (8, 8192, 1024, 2048)


In [5]:

# Question 2.2

mesh = jax.make_mesh(
    (8,),
    ("X",),
    axis_types=(jax.sharding.AxisType.Auto,),
)

shard_W = jax.NamedSharding(mesh, jax.P("X", None, None))
shard_A = jax.NamedSharding(mesh, jax.P("X", None))
shard_B = jax.NamedSharding(mesh, jax.P("X"))
shard_out = jax.NamedSharding(mesh, jax.P("X", None))

with jax.set_mesh(mesh):
    # Small sharded inputs are still useful for quick correctness/debugging.
    W = jax.device_put(W0, shard_W)
    A = jax.device_put(A0, shard_A)
    B = jax.device_put(B0, shard_B)

    # Larger sharded inputs are what we benchmark for 2.2/2.3.1/2.3.2.
    # Generate them directly with sharded outputs so they never need to fit
    # unsharded on one TPU core.
    @jax.jit(out_shardings=(shard_W, shard_A, shard_B))
    def make_big_inputs(key):
        kW, kA, kB = jax.random.split(key, 3)
        W = jax.random.normal(kW, (E, D_big, F_big))
        A = jax.random.normal(kA, (S_big, D_big))
        B = jax.random.randint(kB, (S_big,), 0, E, dtype=jnp.int32)
        return W, A, B

    W_big, A_big, B_big = make_big_inputs(jax.random.key(10))
    jax.block_until_ready((W_big, A_big, B_big))

    moe_jit = jax.jit(
        moe_local,
        out_shardings=shard_out,
    )

    y = moe_jit(W, A, B).block_until_ready()

    np.testing.assert_allclose(
        np.array(y),
        np.array(moe_reference(W0, A0, B0)),
        rtol=1e-1,
        atol=1e-1,
    )

    # Compile/check the large shape once so timing later does not include compile.
    y_big = moe_jit(W_big, A_big, B_big).block_until_ready()
    print("2.2 large output shape:", y_big.shape)

    compiled = moe_jit.lower(W_big, A_big, B_big).compile()
    hlo = compiled.as_text()

    collectives = []
    for line in hlo.splitlines():
        if any(x in line.lower() for x in [
            "all-gather",
            "all_gather",
            "all-to-all",
            "all_to_all",
            "collective-permute",
            "reduce-scatter",
            "all-reduce",
        ]):
            collectives.append(line)
            print(line)


2.2 large output shape: (8192, 2048)
  %collective-permute-start.6 = (f32[1,1024,2048]{2,1,0:T(8,128)}, f32[1,1024,2048]{2,1,0:T(8,128)S(1)}, u32[]{:S(2)}, u32[]{:S(2)}) collective-permute-start(%param.2), channel_id=14, source_target_pairs={{7,0}}, metadata={op_name="jit(moe_local)/slice" stack_frame_id=20}, backend_config={"barrier_config":{"barrier_type":"CUSTOM","id":"7"},"flag_configs":[],"scoped_memory_configs":[],"used_scoped_memory_configs":[]}
  %collective-permute-start.5 = (f32[1,1024,2048]{2,1,0:T(8,128)}, f32[1,1024,2048]{2,1,0:T(8,128)S(1)}, u32[]{:S(2)}, u32[]{:S(2)}) collective-permute-start(%param.2), channel_id=12, source_target_pairs={{6,0}}, metadata={op_name="jit(moe_local)/slice" stack_frame_id=20}, backend_config={"barrier_config":{"barrier_type":"CUSTOM","id":"6"},"flag_configs":[],"scoped_memory_configs":[],"used_scoped_memory_configs":[]}
  %collective-permute-start.4 = (f32[1,1024,2048]{2,1,0:T(8,128)}, f32[1,1024,2048]{2,1,0:T(8,128)S(1)}, u32[]{:S(2)}, u32[

In [7]:
# Question 2.2 profile

import os
import time

profile_dir = os.path.join("./tensorboard", f"q2_2_moe_jit_{int(time.time())}")
os.makedirs(profile_dir, exist_ok=True)

# Use the large sharded inputs if you ran the updated 2.2 cell. Fall back to
# the smaller inputs if only the original 2.2 setup has been executed.
try:
    profile_args = (W_big, A_big, B_big)
    profile_shape = A_big.shape
except NameError:
    profile_args = (W, A, B)
    profile_shape = A.shape

with jax.set_mesh(mesh):
    # Compile and warm up outside the trace so TensorBoard focuses on steady-state execution.
    for _ in range(3):
        jax.block_until_ready(moe_jit(*profile_args))

    with jax.profiler.trace(profile_dir):
        for step in range(10):
            with jax.profiler.StepTraceAnnotation("moe_jit", step_num=step):
                y_profile = moe_jit(*profile_args)
                jax.block_until_ready(y_profile)

print("profiled moe_jit input A shape:", profile_shape)
print("wrote trace to:", profile_dir)
print("open TensorBoard with: %tensorboard --logdir ./tensorboard")

profiled moe_jit input A shape: (8192, 1024)
wrote trace to: ./tensorboard/q2_2_moe_jit_1779825953
open TensorBoard with: %tensorboard --logdir ./tensorboard


In [8]:
# Question 2.3.1


def moe_shmap_first_pass(W: jnp.ndarray, A: jnp.ndarray, B: jnp.ndarray) -> jnp.ndarray:
    def local_moe(W_local, A_local, B_local):
        A_full = jax.lax.all_gather(A_local, "X", axis=0, tiled=True)  # [S, D]
        B_full = jax.lax.all_gather(B_local, "X", axis=0, tiled=True)  # [S]

        E_local = W_local.shape[0]
        S_local = A_local.shape[0]
        F = W_local.shape[2]

        local_expert_start = jax.lax.axis_index("X") * E_local
        local_expert_ids = local_expert_start + jnp.arange(E_local, dtype=B_full.dtype)

        local_all_out = jnp.einsum("sd,edf->sef", A_full, W_local)  # [S, E_X, F]
        local_mask = (B_full[:, None] == local_expert_ids[None, :])[:, :, None]
        local_contrib = jnp.sum(local_all_out * local_mask, axis=1)  # [S, F]

        full_out = jax.lax.psum(local_contrib, "X")

        token_start = jax.lax.axis_index("X") * S_local
        return jax.lax.dynamic_slice(full_out, (token_start, 0), (S_local, F))

    return jax.shard_map(
        local_moe,
        mesh=mesh,
        in_specs=(jax.P("X", None, None), jax.P("X", None), jax.P("X")),
        out_specs=jax.P("X", None),
        check_vma=False,
    )(W, A, B)


with jax.set_mesh(mesh):
    y_shmap = moe_shmap_first_pass(W, A, B).block_until_ready()

    np.testing.assert_allclose(
        np.array(y_shmap),
        np.array(moe_reference(W0, A0, B0)),
        rtol=1e-1,
        atol=1e-1,
    )

    moe_shmap_jit = jax.jit(
        moe_shmap_first_pass,
        out_shardings=shard_out,
    )

    y_shmap_jit = moe_shmap_jit(W, A, B).block_until_ready()
    np.testing.assert_allclose(
        np.array(y_shmap_jit),
        np.array(moe_reference(W0, A0, B0)),
        rtol=1e-1,
        atol=1e-1,
    )

    y_shmap_big = moe_shmap_jit(W_big, A_big, B_big).block_until_ready()
    print("2.3.1 large output shape:", y_shmap_big.shape)

    hlo_shmap = moe_shmap_jit.lower(W_big, A_big, B_big).compile().as_text()

    collectives_shmap = []
    for line in hlo_shmap.splitlines():
        if any(x in line.lower() for x in [
            "all-gather",
            "all_gather",
            "all-to-all",
            "collective-permute",
            "reduce-scatter",
            "all-reduce",
        ]):
            collectives_shmap.append(line)
            print(line)

    assert any("all-gather" in line.lower() or "all_gather" in line.lower() for line in collectives_shmap)


2.3.1 large output shape: (8192, 2048)
  %all-gather.8 = s32[8192]{0:T(1024)S(1)} all-gather(%param.5), channel_id=1, replica_groups={{0,1,2,3,4,5,6,7}}, dimensions={0}, use_global_device_ids=true, frontend_attributes={async_collective_name="all-gather-start.1"}, metadata={op_name="jit(moe_shmap_first_pass)/shard_map/all_gather" stack_frame_id=24}, backend_config={"aliasing_operands":{"lists":[]},"barrier_config":{"barrier_type":"CUSTOM","id":"0"},"collective_algorithm_config":{"debug":"\ngroup_size = 8 \nhas_reordering_map: true \nper_stride_size = 4096 bytes \nshard_size = 4096 bytes ","emitter":"1DAllGatherOnMajorDim"},"flag_configs":[],"retry_config":{"retry_count":"0"},"scoped_memory_configs":[],"used_scoped_memory_configs":[{"memory_space":"1","offset":"0","size":"0"}]}
  %compare_convert_fusion = f32[8192]{0:T(1024)S(1)} fusion(%all-gather.8, %axis_index.16), kind=kLoop, calls=%fused_computation.2, metadata={op_name="jit(moe_shmap_first_pass)/shard_map/convert_element_type" stac

In [9]:
# Question 2.3.2

import re

# Use a larger chunk for the larger sharded benchmark. Smaller chunks create
# many tiny all_to_all calls and can hide the benefit we are trying to measure.
chunk_size = 128


@jax.shard_map(
    mesh=mesh,
    in_specs=(jax.P("X", None, None), jax.P("X", None), jax.P("X")),
    out_specs=jax.P("X", None),
    check_vma=False,
)
def sharded_moe_ragged_chunked(
    W_local: jnp.ndarray,
    A_local: jnp.ndarray,
    B_local: jnp.ndarray,
) -> jnp.ndarray:
    # A_local: [S_X, D]
    # B_local: [S_X]
    # W_local: [E_X, D, F]
    local_S = A_local.shape[0]
    E_local, D, F = W_local.shape

    # Sort local tokens by global expert assignment. This gives a local ragged
    # layout: one contiguous region per global expert.
    sort_indices = jnp.argsort(B_local, axis=0)
    unsort_indices = jnp.argsort(sort_indices, axis=0)
    sorted_A = A_local[sort_indices]

    # sizes[e] is the number of this shard's local tokens routed to global expert e.
    sizes = jnp.bincount(B_local, length=E).astype(jnp.int32)
    offsets = jnp.cumsum(sizes, axis=0) - sizes

    # Every shard must run the same number of collective calls.
    max_expert_size = jax.lax.pmax(sizes.max(), axis_name="X")
    num_chunks = (max_expert_size + chunk_size - 1) // chunk_size

    def get_chunk(chunk_idx):
        internal_offsets = chunk_idx * chunk_size + jnp.arange(chunk_size, dtype=jnp.int32)
        indices = offsets[:, None] + internal_offsets[None, :]  # [E, chunk_size]
        mask = internal_offsets[None, :] < sizes[:, None]       # [E, chunk_size]

        # Invalid padded rows read as zeros. `local_S` is out of bounds for
        # sorted_A, and mode="fill" turns those reads into fill_value.
        safe_indices = jnp.where(mask, indices, local_S)
        return sorted_A.at[safe_indices].get(mode="fill", fill_value=0.0)

    def loop_body(chunk_idx, output):
        # [E, chunk_size, D]: for each global expert, this shard's next chunk
        # of local tokens routed to that expert.
        chunk = get_chunk(chunk_idx)

        # Route expert chunks to the devices that own those experts:
        # [E, chunk_size, D] -> [E_X, X * chunk_size, D]
        with jax.named_scope("route_tokens_to_experts"):
            chunk = jax.lax.all_to_all(
                chunk,
                axis_name="X",
                split_axis=0,
                concat_axis=1,
                tiled=True,
            )

        # Local expert computation.
        result = jnp.einsum("ecd,edf->ecf", chunk, W_local)

        # Send results back to the token-owning shards:
        # [E_X, X * chunk_size, F] -> [E, chunk_size, F]
        with jax.named_scope("return_results_to_tokens"):
            result = jax.lax.all_to_all(
                result,
                axis_name="X",
                split_axis=1,
                concat_axis=0,
                tiled=True,
            )

        internal_offsets = chunk_idx * chunk_size + jnp.arange(chunk_size, dtype=jnp.int32)
        indices = offsets[:, None] + internal_offsets[None, :]
        mask = internal_offsets[None, :] < sizes[:, None]

        # Invalid padded rows write out of bounds and are dropped.
        indices = jnp.where(mask, indices, local_S)
        return output.at[indices.reshape(-1)].set(
            result.reshape(-1, F),
            mode="drop",
        )

    output = jnp.zeros((local_S, F), dtype=A_local.dtype)
    output = jax.lax.fori_loop(0, num_chunks, loop_body, output)

    # Undo the initial local sort.
    return output[unsort_indices]


with jax.set_mesh(mesh):
    y_ragged_chunked = sharded_moe_ragged_chunked(W, A, B).block_until_ready()

    np.testing.assert_allclose(
        np.array(y_ragged_chunked),
        np.array(moe_reference(W0, A0, B0)),
        rtol=1e-1,
        atol=1e-1,
    )

    ragged_chunked_jit = jax.jit(
        sharded_moe_ragged_chunked,
        out_shardings=shard_out,
    )

    y_ragged_chunked_jit = ragged_chunked_jit(W, A, B).block_until_ready()
    np.testing.assert_allclose(
        np.array(y_ragged_chunked_jit),
        np.array(moe_reference(W0, A0, B0)),
        rtol=1e-1,
        atol=1e-1,
    )

    y_ragged_chunked_big = ragged_chunked_jit(W_big, A_big, B_big).block_until_ready()
    print("2.3.2 large output shape:", y_ragged_chunked_big.shape)

    hlo_ragged_chunked = ragged_chunked_jit.lower(W_big, A_big, B_big).compile().as_text()

    collective_re = re.compile(
        r"=.*\b(all-gather|all-to-all|all-reduce|collective-permute|reduce-scatter)\(",
        re.IGNORECASE,
    )
    collectives_ragged_chunked = [
        line for line in hlo_ragged_chunked.splitlines() if collective_re.search(line)
    ]

    for line in collectives_ragged_chunked[:8]:
        print(line)

    assert any("all-to-all" in line.lower() for line in collectives_ragged_chunked)
    assert not any("all-gather" in line.lower() for line in collectives_ragged_chunked)


2.3.2 large output shape: (8192, 2048)
  %all_to_all.6 = bf16[8,128,1024]{2,1,0:T(8,128)(2,1)S(1)} all-to-all(%broadcast_select_fusion.2), channel_id=1, replica_groups={{0,1,2,3,4,5,6,7}}, dimensions={0}, metadata={op_name="jit(sharded_moe_ragged_chunked)/shard_map/while/body/route_tokens_to_experts/all_to_all" stack_frame_id=48}, backend_config={"aliasing_operands":{"lists":[]},"barrier_config":{"barrier_type":"GLOBAL","id":"-1"},"collective_algorithm_config":{"debug":"\nmax_rdma_size_bytes: 32768","emitter":"AllToAllEmitter"},"flag_configs":[],"retry_config":{"retry_count":"0"},"scoped_memory_configs":[],"used_scoped_memory_configs":[{"memory_space":"1","offset":"0","size":"0"}]}
  %all_to_all.7 = f32[1,1024,2048]{2,0,1:T(1,128)S(1)} all-to-all(%copy.10), channel_id=1, replica_groups={{0,1,2,3,4,5,6,7}}, dimensions={1}, metadata={op_name="jit(sharded_moe_ragged_chunked)/shard_map/while/body/return_results_to_tokens/all_to_all" stack_frame_id=49}, backend_config={"aliasing_operands":{

In [10]:

# Question 2 timing comparison

import time


def benchmark(label, fn, *args, warmup=3, iters=50):
    for _ in range(warmup):
        jax.block_until_ready(fn(*args))

    start = time.perf_counter()
    for _ in range(iters):
        jax.block_until_ready(fn(*args))
    elapsed = time.perf_counter() - start

    ms = 1000 * elapsed / iters
    print(f"{label:40s} {ms:8.3f} ms/call")
    return ms


with jax.set_mesh(mesh):
    moe_local_jit = jax.jit(moe_local)

    print("Small single-device baseline config:", (E, S, D, F))
    print("Large sharded benchmark config:    ", (E, S_big, D_big, F_big))

    timings = {
        "2.1 local jit small": benchmark(
            "2.1 local jit small",
            moe_local_jit,
            W0,
            A0,
            B0,
        ),
        "2.2 jit(moe_local) sharded big": benchmark(
            "2.2 jit(moe_local) sharded big",
            moe_jit,
            W_big,
            A_big,
            B_big,
            iters=20,
        ),
        "2.3.1 shard_map all_gather big": benchmark(
            "2.3.1 shard_map all_gather big",
            moe_shmap_jit,
            W_big,
            A_big,
            B_big,
            iters=20,
        ),
        "2.3.2 shard_map all_to_all big": benchmark(
            "2.3.2 shard_map all_to_all big",
            ragged_chunked_jit,
            W_big,
            A_big,
            B_big,
            iters=20,
        ),
    }


Small single-device baseline config: (8, 1024, 512, 512)
Large sharded benchmark config:     (8, 8192, 1024, 2048)
2.1 local jit small                         2.050 ms/call
2.2 jit(moe_local) sharded big              2.598 ms/call
2.3.1 shard_map all_gather big              2.048 ms/call
2.3.2 shard_map all_to_all big              1.143 ms/call


In [11]:
# Question 2.4

K = 2

key = jax.random.key(4)
k_topk = jax.random.split(key, 1)[0]
scores = jax.random.uniform(k_topk, (S, E))
B0_topk = jnp.argsort(scores, axis=1)[:, :K].astype(jnp.int32)
B_topk = jax.device_put(B0_topk, jax.NamedSharding(mesh, jax.P("X", None)))


def moe_reference_topk(W, A, B):
    per_route = jax.vmap(
        lambda a, experts: jax.vmap(
            lambda e: jnp.matmul(a, W[e], precision=jax.lax.Precision.HIGHEST)
        )(experts)
    )(A, B)
    return per_route.mean(axis=1)


@jax.shard_map(
    mesh=mesh,
    in_specs=(jax.P("X", None, None), jax.P("X", None), jax.P("X", None)),
    out_specs=jax.P("X", None),
    check_vma=False,
)
def sharded_moe_topk_ragged_chunked(
    W_local: jnp.ndarray,
    A_local: jnp.ndarray,
    B_local: jnp.ndarray,
) -> jnp.ndarray:
    # A_local: [S_X, D]
    # B_local: [S_X, K]
    # W_local: [E_X, D, F]
    local_S, D = A_local.shape
    E_local, _, F = W_local.shape
    local_K = B_local.shape[1]
    local_items = local_S * local_K

    # Expand each token into K routed items.
    A_items = jnp.repeat(A_local, local_K, axis=0)       # [S_X * K, D]
    B_items = B_local.reshape(-1)                       # [S_X * K]
    token_items = jnp.repeat(jnp.arange(local_S, dtype=jnp.int32), local_K)

    # Sort routed items by global expert id to make a ragged layout.
    sort_indices = jnp.argsort(B_items, axis=0)
    sorted_A = A_items[sort_indices]
    sorted_token_items = token_items[sort_indices]

    sizes = jnp.bincount(B_items, length=E).astype(jnp.int32)
    offsets = jnp.cumsum(sizes, axis=0) - sizes

    max_expert_size = jax.lax.pmax(sizes.max(), axis_name="X")
    num_chunks = (max_expert_size + chunk_size - 1) // chunk_size

    def get_chunk(chunk_idx):
        internal_offsets = chunk_idx * chunk_size + jnp.arange(chunk_size, dtype=jnp.int32)
        indices = offsets[:, None] + internal_offsets[None, :]
        mask = internal_offsets[None, :] < sizes[:, None]
        safe_indices = jnp.where(mask, indices, local_items)

        chunk_A = sorted_A.at[safe_indices].get(mode="fill", fill_value=0.0)
        chunk_tokens = sorted_token_items.at[safe_indices].get(mode="fill", fill_value=0)
        return chunk_A, chunk_tokens, mask

    def loop_body(chunk_idx, output):
        chunk_A, chunk_tokens, mask = get_chunk(chunk_idx)

        # Route activations and their original token positions to expert owners.
        with jax.named_scope("route_topk_tokens_to_experts"):
            chunk_A = jax.lax.all_to_all(
                chunk_A,
                axis_name="X",
                split_axis=0,
                concat_axis=1,
                tiled=True,
            )
            chunk_tokens = jax.lax.all_to_all(
                chunk_tokens,
                axis_name="X",
                split_axis=0,
                concat_axis=1,
                tiled=True,
            )
            mask = jax.lax.all_to_all(
                mask,
                axis_name="X",
                split_axis=0,
                concat_axis=1,
                tiled=True,
            )

        result = jnp.einsum(
                    "ecd,edf->ecf",
                    chunk_A,
                    W_local,
                    precision=jax.lax.Precision.HIGHEST,
                )

        # Send results and token positions back to the token-owning shards.
        with jax.named_scope("return_topk_results_to_tokens"):
            result = jax.lax.all_to_all(
                result,
                axis_name="X",
                split_axis=1,
                concat_axis=0,
                tiled=True,
            )
            chunk_tokens = jax.lax.all_to_all(
                chunk_tokens,
                axis_name="X",
                split_axis=1,
                concat_axis=0,
                tiled=True,
            )
            mask = jax.lax.all_to_all(
                mask,
                axis_name="X",
                split_axis=1,
                concat_axis=0,
                tiled=True,
            )

        # Multiple routed experts contribute to the same token, so scatter-add.
        flat_tokens = jnp.where(mask.reshape(-1), chunk_tokens.reshape(-1), local_S)
        flat_result = result.reshape(-1, F)
        return output.at[flat_tokens].add(flat_result, mode="drop")

    output = jnp.zeros((local_S, F), dtype=A_local.dtype)
    output = jax.lax.fori_loop(0, num_chunks, loop_body, output)
    return output / local_K


with jax.set_mesh(mesh):
    y_topk = sharded_moe_topk_ragged_chunked(W, A, B_topk).block_until_ready()

    np.testing.assert_allclose(
        np.array(y_topk),
        np.array(moe_reference_topk(W0, A0, B0_topk)),
        rtol=1e-1,
        atol=1e-1,
    )

    topk_jit = jax.jit(
        sharded_moe_topk_ragged_chunked,
        out_shardings=shard_out,
    )

    y_topk_jit = topk_jit(W, A, B_topk).block_until_ready()
    np.testing.assert_allclose(
        np.array(y_topk_jit),
        np.array(moe_reference_topk(W0, A0, B0_topk)),
        rtol=1e-1,
        atol=1e-1,
    )

    hlo_topk = topk_jit.lower(W, A, B_topk).compile().as_text()

    collective_re = re.compile(
        r"=.*\b(all-gather|all-to-all|all-reduce|collective-permute|reduce-scatter)\(",
        re.IGNORECASE,
    )
    collectives_topk = [
        line for line in hlo_topk.splitlines() if collective_re.search(line)
    ]

    for line in collectives_topk[:10]:
        print(line)

    assert any("all-to-all" in line.lower() for line in collectives_topk)
    assert not any("all-gather" in line.lower() for line in collectives_topk)


  %all_to_all.28 = s32[8,1,128]{2,1,0:T(1,128)S(1)} all-to-all(%and_select_fusion.2), channel_id=1, replica_groups={{0,1,2,3,4,5,6,7}}, dimensions={0}, metadata={op_name="jit(sharded_moe_topk_ragged_chunked)/shard_map/while/body/route_topk_tokens_to_experts/all_to_all" stack_frame_id=59}, backend_config={"aliasing_operands":{"lists":[]},"barrier_config":{"barrier_type":"GLOBAL","id":"-1"},"collective_algorithm_config":{"debug":"\nmax_rdma_size_bytes: 32768","emitter":"AllToAllEmitter"},"flag_configs":[],"retry_config":{"retry_count":"0"},"scoped_memory_configs":[],"used_scoped_memory_configs":[{"memory_space":"1","offset":"0","size":"0"}]}
  %all_to_all.29 = s32[1,8,128]{2,0,1:T(1,128)S(1)} all-to-all(%bitcast.53), channel_id=1, replica_groups={{0,1,2,3,4,5,6,7}}, dimensions={1}, metadata={op_name="jit(sharded_moe_topk_ragged_chunked)/shard_map/while/body/return_topk_results_to_tokens/all_to_all" stack_frame_id=60}, backend_config={"aliasing_operands":{"lists":[]},"barrier_config":{"ba

In [12]:
# Question 3.1 scaffold

# 2D mesh for the collective matmul exercises:
#   X shards the batch/token dimension B
#   Y shards the contraction/hidden dimension D
mesh_shape_env = os.environ.get("SECTION10_MESH")
if mesh_shape_env is None:
    mesh_x = 2 if jax.device_count() >= 4 and jax.device_count() % 2 == 0 else 1
    mesh_shape = (mesh_x, jax.device_count() // mesh_x)
else:
    mesh_shape = tuple(int(x) for x in mesh_shape_env.replace("x", ",").split(","))

assert len(mesh_shape) == 2
assert mesh_shape[0] * mesh_shape[1] == jax.device_count(), (
    f"mesh_shape={mesh_shape} must use exactly {jax.device_count()} devices"
)

mesh_2d = jax.make_mesh(
    mesh_shape,
    ("X", "Y"),
    axis_types=(jax.sharding.AxisType.Auto, jax.sharding.AxisType.Auto),
)

# For Q3.1, communication is [B_X, F] while matmul compute scales with D_Y.
# A larger contracting dimension D gives the tiled collectives more compute to
# hide behind. Override with, e.g.:
#   SECTION10_Q31_B=8192 SECTION10_Q31_D=16384 SECTION10_Q31_F=8192
Q31_ON_TPU = jax.default_backend() == "tpu"
Q31_DEFAULT_B = 8192 if Q31_ON_TPU else 1024
Q31_DEFAULT_D = 16384 if Q31_ON_TPU else 1024
Q31_DEFAULT_F = 8192 if Q31_ON_TPU else 4096

B3 = int(os.environ.get("SECTION10_Q31_B", Q31_DEFAULT_B))
D3 = int(os.environ.get("SECTION10_Q31_D", Q31_DEFAULT_D))
F3 = int(os.environ.get("SECTION10_Q31_F", Q31_DEFAULT_F))
Q31_NUM_F_TILES = int(os.environ.get("SECTION10_Q31_F_TILES", str(mesh_2d.shape["Y"])))

assert B3 % mesh_2d.shape["X"] == 0
assert D3 % mesh_2d.shape["Y"] == 0
assert F3 % Q31_NUM_F_TILES == 0

F3_tile = F3 // Q31_NUM_F_TILES

Q31_MATMUL_DTYPE = jnp.bfloat16 if Q31_ON_TPU else jnp.float32
Q31_REDUCE_DTYPE = jnp.bfloat16 if Q31_ON_TPU else jnp.float32
Q31_RTOL = float(os.environ.get("SECTION10_Q31_RTOL", "5e-2"))
Q31_ATOL = float(os.environ.get("SECTION10_Q31_ATOL", "32.0" if Q31_ON_TPU else "2.0"))

key = jax.random.key(31)
kA, kW = jax.random.split(key, 2)

A3_full = jax.random.normal(kA, (B3, D3), dtype=Q31_MATMUL_DTYPE)
W3_full = jax.random.normal(kW, (D3, F3), dtype=Q31_MATMUL_DTYPE)


def allreduce_matmul_reference(A: jnp.ndarray, W: jnp.ndarray) -> jnp.ndarray:
    return A @ W


with jax.set_mesh(mesh_2d):
    A3 = jax.device_put(A3_full, jax.NamedSharding(mesh_2d, jax.P("X", "Y")))
    W3 = jax.device_put(W3_full, jax.NamedSharding(mesh_2d, jax.P("Y", None)))

    A3_sharding = jax.NamedSharding(mesh_2d, jax.P("X", "Y"))
    W3_sharding = jax.NamedSharding(mesh_2d, jax.P("Y", None))
    out3_sharding = jax.NamedSharding(mesh_2d, jax.P("X", None))


@jax.shard_map(
    mesh=mesh_2d,
    in_specs=(jax.P("X", "Y"), jax.P("Y", None)),
    out_specs=jax.P("X", None),
    check_vma=False,
)
def allreduce_matmul_baseline(A_local: jnp.ndarray, W_local: jnp.ndarray) -> jnp.ndarray:
    # A_local: [B_X, D_Y]
    # W_local: [D_Y, F]
    # Return:  [B_X, F]
    out_local = (A_local @ W_local).astype(Q31_REDUCE_DTYPE)
    out = jax.lax.psum(out_local, axis_name="Y")
    return out.astype(A_local.dtype)


@jax.shard_map(
    mesh=mesh_2d,
    in_specs=(jax.P("X", "Y"), jax.P("Y", None)),
    out_specs=jax.P("X", None),
    check_vma=False,
)
def allreduce_matmul_tiled(A_local: jnp.ndarray, W_local: jnp.ndarray) -> jnp.ndarray:
    # A_local: [B_X, D_Y]
    # W_local: [D_Y, F]
    # Return:  [B_X, F]
    B_X, D_Y = A_local.shape
    _, F = W_local.shape

    out = jnp.zeros((B_X, F), dtype=Q31_REDUCE_DTYPE)

    for tile_idx in range(Q31_NUM_F_TILES):
        f0 = tile_idx * F3_tile
        W_tile = jax.lax.dynamic_slice(W_local, (0, f0), (D_Y, F3_tile))
        partial = (A_local @ W_tile).astype(Q31_REDUCE_DTYPE)
        tile = jax.lax.psum(partial, axis_name="Y")
        out = jax.lax.dynamic_update_slice(out, tile, start_indices=(0, f0))

    return out.astype(A_local.dtype)


def test_allreduce_matmul_impls():
    with jax.set_mesh(mesh_2d):
        plain_jit = jax.jit(allreduce_matmul_reference, out_shardings=out3_sharding)
        baseline_jit = jax.jit(allreduce_matmul_baseline, out_shardings=out3_sharding)
        tiled_jit = jax.jit(allreduce_matmul_tiled, out_shardings=out3_sharding)

        y_plain = plain_jit(A3, W3).block_until_ready()
        y_baseline = baseline_jit(A3, W3).block_until_ready()
        y_tiled = tiled_jit(A3, W3).block_until_ready()

        np.testing.assert_allclose(np.array(y_baseline), np.array(y_plain), rtol=Q31_RTOL, atol=Q31_ATOL)
        np.testing.assert_allclose(np.array(y_tiled), np.array(y_baseline), rtol=Q31_RTOL, atol=Q31_ATOL)

        hlo_baseline = baseline_jit.lower(A3, W3).compile().as_text()
        hlo_tiled = tiled_jit.lower(A3, W3).compile().as_text()

        print("baseline collectives:")
        for line in hlo_baseline.splitlines():
            if any(x in line.lower() for x in ["all-reduce", "all-gather", "all-to-all", "collective-permute"]):
                print(line)

        print("tiled collectives:")
        for line in hlo_tiled.splitlines():
            if any(x in line.lower() for x in ["all-reduce", "all-gather", "all-to-all", "collective-permute"]):
                print(line)


# This is intentionally commented because the TPU-sized defaults are large.
test_allreduce_matmul_impls()


baseline collectives:
  ROOT %psum.7 = bf16[4096,8192]{1,0:T(8,128)(2,1)} all-reduce(%fusion), channel_id=1, replica_groups={{0,1,2,3},{4,5,6,7}}, use_global_device_ids=true, to_apply=%region_0.0, metadata={op_name="jit(allreduce_matmul_baseline)/shard_map/psum" stack_frame_id=24}, backend_config={"aliasing_operands":{"lists":[{"indices":["0","1"]}]},"barrier_config":{"barrier_type":"CUSTOM","id":"0"},"collective_algorithm_config":{"debug":"\nUniDirection1DRingStrategy{colors:2 phases:1 cores:{4},{4} nophase0:0 reserved_sflags:0 cross_module_on_2d_plane:0 has_reordering_map:0 use_routing_table_indices:0}","emitter":"RotatedPincerEmitter","strategy":"UniDirection1DRingStrategy"},"flag_configs":[],"retry_config":{"retry_count":"0"},"scoped_memory_configs":[{"memory_space":"0","offset":"0","size":"67108864"}],"used_scoped_memory_configs":[{"memory_space":"1","offset":"0","size":"14942208"}]}
tiled collectives:
  %all-reduce.1 = (bf16[4096,2048]{1,0:T(8,128)(2,1)S(1)}, bf16[4096,2048]{1,0:

In [13]:
# Question 3.1 timing comparison

import time


def benchmark_q31(label, fn, *args, warmup=5, iters=20):
    for _ in range(warmup):
        jax.block_until_ready(fn(*args))

    start = time.perf_counter()
    for _ in range(iters):
        jax.block_until_ready(fn(*args))
    elapsed = time.perf_counter() - start

    per_call_ms = elapsed / iters * 1e3
    print(f"{label:<36} {per_call_ms:8.3f} ms/call")
    return per_call_ms


with jax.set_mesh(mesh_2d):
    q31_warmup = int(os.environ.get("SECTION10_Q31_WARMUP", "5"))
    q31_iters = int(os.environ.get("SECTION10_Q31_ITERS", "20" if Q31_ON_TPU else "10"))

    plain_ar_jit = jax.jit(allreduce_matmul_reference, out_shardings=out3_sharding)
    baseline_ar_jit = jax.jit(allreduce_matmul_baseline, out_shardings=out3_sharding)
    tiled_ar_jit = jax.jit(allreduce_matmul_tiled, out_shardings=out3_sharding)

    print("Question 3.1 config:")
    print("  backend:", jax.default_backend())
    print("  mesh:", mesh_2d)
    print("  shapes:", {"A": A3.shape, "W": W3.shape, "out": (B3, F3)})
    print("  F tile:", {"num_tiles": Q31_NUM_F_TILES, "tile_size": F3_tile})
    print("  matmul dtype:", Q31_MATMUL_DTYPE)
    print("  reduce dtype:", Q31_REDUCE_DTYPE)
    print("  tolerances:", {"rtol": Q31_RTOL, "atol": Q31_ATOL})
    print("  benchmark:", {"warmup": q31_warmup, "iters": q31_iters})

    y_plain = plain_ar_jit(A3, W3).block_until_ready()
    y_baseline = baseline_ar_jit(A3, W3).block_until_ready()
    y_tiled = tiled_ar_jit(A3, W3).block_until_ready()

    np.testing.assert_allclose(np.array(y_baseline), np.array(y_plain), rtol=Q31_RTOL, atol=Q31_ATOL)
    np.testing.assert_allclose(np.array(y_tiled), np.array(y_baseline), rtol=Q31_RTOL, atol=Q31_ATOL)

    q31_plain_ms = benchmark_q31(
        "3.1 plain jax.jit baseline",
        plain_ar_jit,
        A3,
        W3,
        warmup=q31_warmup,
        iters=q31_iters,
    )
    q31_baseline_ms = benchmark_q31(
        "3.1 shard_map psum",
        baseline_ar_jit,
        A3,
        W3,
        warmup=q31_warmup,
        iters=q31_iters,
    )
    q31_tiled_ms = benchmark_q31(
        "3.1 tiled psum bf16",
        tiled_ar_jit,
        A3,
        W3,
        warmup=q31_warmup,
        iters=q31_iters,
    )

    print(f"tiled vs plain jit speedup:       {q31_plain_ms / q31_tiled_ms:8.3f}x")
    print(f"tiled vs shard_map psum:         {q31_baseline_ms / q31_tiled_ms:8.3f}x")


Question 3.1 config:
  backend: tpu
  mesh: Mesh('X': 2, 'Y': 4, axis_types=(Auto, Auto))
  shapes: {'A': (8192, 16384), 'W': (16384, 8192), 'out': (8192, 8192)}
  F tile: {'num_tiles': 4, 'tile_size': 2048}
  matmul dtype: <class 'jax.numpy.bfloat16'>
  reduce dtype: <class 'jax.numpy.bfloat16'>
  tolerances: {'rtol': 0.05, 'atol': 32.0}
  benchmark: {'warmup': 5, 'iters': 20}
3.1 plain jax.jit baseline              2.990 ms/call
3.1 shard_map psum                      2.991 ms/call
3.1 tiled psum bf16                     3.167 ms/call
tiled vs plain jit speedup:          0.944x
tiled vs shard_map psum:            0.945x


In [14]:
# Question 3.2 scaffold

# Reuse mesh_2d from Question 3.1:
#   X shards batch/tokens B
#   Y shards the intermediate dimension F and the output dimension D
#
# TPU defaults are intentionally large enough that ppermute traffic can be
# hidden behind the matmul. Override with, e.g.:
#   SECTION10_Q32_B=4096 SECTION10_Q32_F=8192 SECTION10_Q32_D=4096
Q32_ON_TPU = jax.default_backend() == "tpu"
Q32_DEFAULT_B = 8192 if Q32_ON_TPU else 1024
Q32_DEFAULT_F = 16384 if Q32_ON_TPU else 4096
Q32_DEFAULT_D = 8192 if Q32_ON_TPU else 1024

B4 = int(os.environ.get("SECTION10_Q32_B", Q32_DEFAULT_B))
F4 = int(os.environ.get("SECTION10_Q32_F", Q32_DEFAULT_F))
D4 = int(os.environ.get("SECTION10_Q32_D", Q32_DEFAULT_D))

assert B4 % mesh_2d.shape["X"] == 0
assert F4 % mesh_2d.shape["Y"] == 0
assert D4 % mesh_2d.shape["Y"] == 0

D4_tile = D4 // mesh_2d.shape["Y"]

Q32_MATMUL_DTYPE = jnp.bfloat16 if Q32_ON_TPU else jnp.float32
Q32_RING_ACCUM_DTYPE = jnp.bfloat16
Q32_RTOL = 5e-2
Q32_ATOL = 8

key = jax.random.key(32)
kT, kW2 = jax.random.split(key, 2)

Tmp4_full = jax.random.normal(kT, (B4, F4), dtype=Q32_MATMUL_DTYPE)
W4_full = jax.random.normal(kW2, (F4, D4), dtype=Q32_MATMUL_DTYPE)


def reducescatter_matmul_reference(Tmp: jnp.ndarray, W2: jnp.ndarray) -> jnp.ndarray:
    return Tmp @ W2


with jax.set_mesh(mesh_2d):
    Tmp4 = jax.device_put(Tmp4_full, jax.NamedSharding(mesh_2d, jax.P("X", "Y")))
    W4 = jax.device_put(W4_full, jax.NamedSharding(mesh_2d, jax.P("Y", None)))

    Tmp4_sharding = jax.NamedSharding(mesh_2d, jax.P("X", "Y"))
    W4_sharding = jax.NamedSharding(mesh_2d, jax.P("Y", None))
    out4_sharding = jax.NamedSharding(mesh_2d, jax.P("X", "Y"))


@jax.shard_map(
    mesh=mesh_2d,
    in_specs=(jax.P("X", "Y"), jax.P("Y", None)),
    out_specs=jax.P("X", "Y"),
    check_vma=False,
)
def reducescatter_matmul_baseline(Tmp_local: jnp.ndarray, W2_local: jnp.ndarray) -> jnp.ndarray:
    # Tmp_local: [B_X, F_Y]
    # W2_local:  [F_Y, D]
    # Return:    [B_X, D_Y]
    B_X = Tmp_local.shape[0]

    out_p = Tmp_local @ W2_local
    out = jax.lax.psum(out_p, axis_name="Y")
    d0 = jax.lax.axis_index("Y") * D4_tile
    return jax.lax.dynamic_slice(out, (0, d0), (B_X, D4_tile))


@jax.shard_map(
    mesh=mesh_2d,
    in_specs=(jax.P("X", "Y"), jax.P("Y", None)),
    out_specs=jax.P("X", "Y"),
    check_vma=False,
)
def reducescatter_matmul_tiled(Tmp_local: jnp.ndarray, W2_local: jnp.ndarray) -> jnp.ndarray:
    # Tmp_local: [B_X, F_Y]
    # W2_local:  [F_Y, D]
    # Return:    [B_X, D_Y]
    y = jax.lax.axis_index("Y")
    B_X, F_Y = Tmp_local.shape

    static_Y = mesh_2d.shape["Y"]
    perm = [(i, (i + 1) % static_Y) for i in range(static_Y)]

    def contribution_for_tile(tile_owner):
        d0 = tile_owner * D4_tile
        W_tile = jax.lax.dynamic_slice(
            W2_local,
            (0, d0),
            (F_Y, D4_tile),
        )
        return (Tmp_local @ W_tile).astype(Q32_RING_ACCUM_DTYPE)

    # Start each tile one hop past its final owner. After static_Y - 1 forward
    # permutes, the fully accumulated tile lands on the owning Y device. This
    # removes the old extra return-home ppermute.
    tile_owner = (y - 1) % static_Y
    accum = contribution_for_tile(tile_owner)

    for _ in range(static_Y - 1):
        accum = jax.lax.ppermute(accum, axis_name="Y", perm=perm)
        tile_owner = (tile_owner - 1) % static_Y
        accum = (accum + contribution_for_tile(tile_owner)).astype(Q32_RING_ACCUM_DTYPE)

    return accum.astype(Tmp_local.dtype)


def test_reducescatter_matmul_impls():
    expected = np.array(reducescatter_matmul_reference(Tmp4_full, W4_full))

    with jax.set_mesh(mesh_2d):
        baseline_jit = jax.jit(reducescatter_matmul_baseline, out_shardings=out4_sharding)
        tiled_jit = jax.jit(reducescatter_matmul_tiled, out_shardings=out4_sharding)

        y_baseline = baseline_jit(Tmp4, W4).block_until_ready()
        np.testing.assert_allclose(np.array(y_baseline), expected, rtol=Q32_RTOL, atol=Q32_ATOL)

        y_tiled = tiled_jit(Tmp4, W4).block_until_ready()
        np.testing.assert_allclose(np.array(y_tiled), expected, rtol=Q32_RTOL, atol=Q32_ATOL)

        hlo_baseline = baseline_jit.lower(Tmp4, W4).compile().as_text()
        hlo_tiled = tiled_jit.lower(Tmp4, W4).compile().as_text()

        print("baseline collectives:")
        for line in hlo_baseline.splitlines():
            if any(x in line.lower() for x in ["all-reduce", "all-gather", "all-to-all", "collective-permute"]):
                print(line)

        print("tiled collectives:")
        for line in hlo_tiled.splitlines():
            if any(x in line.lower() for x in ["all-reduce", "all-gather", "all-to-all", "collective-permute"]):
                print(line)


# This is intentionally commented because the TPU-sized defaults are large.
test_reducescatter_matmul_impls()


baseline collectives:
  %psum.7 = bf16[4096,8192]{1,0:T(8,128)(2,1)} all-reduce(%fusion), channel_id=1, replica_groups={{0,1,2,3},{4,5,6,7}}, use_global_device_ids=true, to_apply=%region_0.0, metadata={op_name="jit(reducescatter_matmul_baseline)/shard_map/psum" stack_frame_id=24}, backend_config={"aliasing_operands":{"lists":[{"indices":["0","1"]}]},"barrier_config":{"barrier_type":"CUSTOM","id":"0"},"collective_algorithm_config":{"debug":"\nUniDirection1DRingStrategy{colors:2 phases:1 cores:{4},{4} nophase0:0 reserved_sflags:0 cross_module_on_2d_plane:0 has_reordering_map:0 use_routing_table_indices:0}","emitter":"RotatedPincerEmitter","strategy":"UniDirection1DRingStrategy"},"flag_configs":[],"retry_config":{"retry_count":"0"},"scoped_memory_configs":[{"memory_space":"0","offset":"0","size":"67108864"}],"used_scoped_memory_configs":[{"memory_space":"1","offset":"0","size":"14942208"}]}
tiled collectives:
  %collective-permute-start = (bf16[4096,2048]{1,0:T(8,128)(2,1)S(1)}, bf16[4096

In [15]:
# Question 3.2 timing comparison

import time


def benchmark_q32(label, fn, *args, warmup=5, iters=20):
    for _ in range(warmup):
        jax.block_until_ready(fn(*args))

    start = time.perf_counter()
    for _ in range(iters):
        jax.block_until_ready(fn(*args))
    elapsed = time.perf_counter() - start

    per_call_ms = elapsed / iters * 1e3
    print(f"{label:<36} {per_call_ms:8.3f} ms/call")
    return per_call_ms


with jax.set_mesh(mesh_2d):
    q32_warmup = int(os.environ.get("SECTION10_Q32_WARMUP", "5"))
    q32_iters = int(os.environ.get("SECTION10_Q32_ITERS", "20" if Q32_ON_TPU else "10"))

    baseline_rs_jit = jax.jit(reducescatter_matmul_baseline, out_shardings=out4_sharding)
    tiled_rs_jit = jax.jit(reducescatter_matmul_tiled, out_shardings=out4_sharding)

    print("Question 3.2 config:")
    print("  backend:", jax.default_backend())
    print("  mesh:", mesh_2d)
    print("  shapes:", {"Tmp": Tmp4.shape, "W2": W4.shape, "out": (B4, D4)})
    print("  matmul dtype:", Q32_MATMUL_DTYPE)
    print("  ring accumulator dtype:", Q32_RING_ACCUM_DTYPE)
    print("  benchmark:", {"warmup": q32_warmup, "iters": q32_iters})

    y_baseline = baseline_rs_jit(Tmp4, W4).block_until_ready()
    y_tiled = tiled_rs_jit(Tmp4, W4).block_until_ready()
    np.testing.assert_allclose(
        np.array(y_tiled),
        np.array(y_baseline),
        rtol=Q32_RTOL,
        atol=Q32_ATOL,
    )

    q32_baseline_ms = benchmark_q32(
        "3.2 baseline psum",
        baseline_rs_jit,
        Tmp4,
        W4,
        warmup=q32_warmup,
        iters=q32_iters,
    )
    q32_tiled_ms = benchmark_q32(
        "3.2 ppermute ring bf16",
        tiled_rs_jit,
        Tmp4,
        W4,
        warmup=q32_warmup,
        iters=q32_iters,
    )

    print(f"speedup vs baseline: {q32_baseline_ms / q32_tiled_ms:.3f}x")


Question 3.2 config:
  backend: tpu
  mesh: Mesh('X': 2, 'Y': 4, axis_types=(Auto, Auto))
  shapes: {'Tmp': (8192, 16384), 'W2': (16384, 8192), 'out': (8192, 8192)}
  matmul dtype: <class 'jax.numpy.bfloat16'>
  ring accumulator dtype: <class 'jax.numpy.bfloat16'>
  benchmark: {'warmup': 5, 'iters': 20}
3.2 baseline psum                       3.036 ms/call
3.2 ppermute ring bf16                  2.273 ms/call
speedup vs baseline: 1.336x


In [16]:
# Question 3.3 scaffold

# End-to-end block dimensions. Reuse the same mesh from Questions 3.1 and 3.2:
#   In:    [B_X, D_Y]
#   W_in:  [D_Y, F]
#   Tmp:   [B_X, F_Y]
#   W_out: [F_Y, D]
#   Out:   [B_X, D_Y]
#
# TPU defaults are intentionally large enough that ppermute traffic can be
# hidden behind the matmuls. Override with, e.g.:
#   SECTION10_Q33_B=4096 SECTION10_Q33_D=8192 SECTION10_Q33_F=16384
Q33_ON_TPU = jax.default_backend() == "tpu"
Q33_DEFAULT_B = 8192 if Q33_ON_TPU else 1024
Q33_DEFAULT_D = 16384 if Q33_ON_TPU else 1024
Q33_DEFAULT_F = 8192 if Q33_ON_TPU else 4096

B5 = int(os.environ.get("SECTION10_Q33_B", Q33_DEFAULT_B))
D5 = int(os.environ.get("SECTION10_Q33_D", Q33_DEFAULT_D))
F5 = int(os.environ.get("SECTION10_Q33_F", Q33_DEFAULT_F))

assert B5 % mesh_2d.shape["X"] == 0
assert D5 % mesh_2d.shape["Y"] == 0
assert F5 % mesh_2d.shape["Y"] == 0

D5_tile = D5 // mesh_2d.shape["Y"]
F5_tile = F5 // mesh_2d.shape["Y"]

Q33_MATMUL_DTYPE = jnp.bfloat16 if Q33_ON_TPU else jnp.float32
Q33_RING_ACCUM_DTYPE = jnp.bfloat16
Q33_RTOL = float(os.environ.get("SECTION10_Q33_RTOL", "5e-2"))
Q33_ATOL = float(os.environ.get("SECTION10_Q33_ATOL", "512.0" if Q33_ON_TPU else "2.0"))

key = jax.random.key(33)
kIn, kWin, kWout = jax.random.split(key, 3)

In5_full = jax.random.normal(kIn, (B5, D5), dtype=Q33_MATMUL_DTYPE)
Win5_full = jax.random.normal(kWin, (D5, F5), dtype=Q33_MATMUL_DTYPE)
Wout5_full = jax.random.normal(kWout, (F5, D5), dtype=Q33_MATMUL_DTYPE)


def transformer_block_reference(
    In: jnp.ndarray,
    Win: jnp.ndarray,
    Wout: jnp.ndarray,
) -> jnp.ndarray:
    return (In @ Win) @ Wout


with jax.set_mesh(mesh_2d):
    In5 = jax.device_put(In5_full, jax.NamedSharding(mesh_2d, jax.P("X", "Y")))
    Win5 = jax.device_put(Win5_full, jax.NamedSharding(mesh_2d, jax.P("Y", None)))
    Wout5 = jax.device_put(Wout5_full, jax.NamedSharding(mesh_2d, jax.P("Y", None)))

    In5_sharding = jax.NamedSharding(mesh_2d, jax.P("X", "Y"))
    Win5_sharding = jax.NamedSharding(mesh_2d, jax.P("Y", None))
    Wout5_sharding = jax.NamedSharding(mesh_2d, jax.P("Y", None))
    out5_sharding = jax.NamedSharding(mesh_2d, jax.P("X", "Y"))


@jax.shard_map(
    mesh=mesh_2d,
    in_specs=(jax.P("X", "Y"), jax.P("Y", None), jax.P("Y", None)),
    out_specs=jax.P("X", "Y"),
    check_vma=False,
)
def transformer_block_baseline(
    In_local: jnp.ndarray,
    Win_local: jnp.ndarray,
    Wout_local: jnp.ndarray,
) -> jnp.ndarray:
    # In_local:   [B_X, D_Y]
    # Win_local:  [D_Y, F]
    # Wout_local: [F_Y, D]
    # Return:     [B_X, D_Y]
    tmp_partial = In_local @ Win_local
    tmp = jax.lax.psum(tmp_partial, axis_name="Y")

    y = jax.lax.axis_index("Y")
    B_X, D_Y = In_local.shape
    F_Y, _ = Wout_local.shape

    f0 = y * F_Y
    tmp_slice = jax.lax.dynamic_slice(tmp, (0, f0), (B_X, F_Y))

    out_partial = tmp_slice @ Wout_local
    out = jax.lax.psum(out_partial, axis_name="Y")

    d0 = y * D_Y
    return jax.lax.dynamic_slice(out, (0, d0), (B_X, D_Y))


@jax.shard_map(
    mesh=mesh_2d,
    in_specs=(jax.P("X", "Y"), jax.P("Y", None), jax.P("Y", None)),
    out_specs=jax.P("X", "Y"),
    check_vma=False,
)
def transformer_block_tiled(
    In_local: jnp.ndarray,
    Win_local: jnp.ndarray,
    Wout_local: jnp.ndarray,
) -> jnp.ndarray:
    # In_local:   [B_X, D_Y]
    # Win_local:  [D_Y, F]
    # Wout_local: [F_Y, D]
    # Return:     [B_X, D_Y]
    #
    # Hybrid approach:
    #   1. Use one full-width up-projection psum, then keep this device's
    #      hidden F_Y shard.
    #   2. Use the corrected Q3.2 ppermute ring for the down projection.
    #
    # This avoids the old Y separate hidden psums and Y * (Y - 1) down rings.
    # With Y=4, the communication drops from 4 psums + 12 ppermutes to
    # 1 psum + 3 ppermutes.
    y = jax.lax.axis_index("Y")
    B_X, D_Y = In_local.shape
    F_Y, _ = Wout_local.shape

    static_Y = mesh_2d.shape["Y"]
    perm = [(i, (i + 1) % static_Y) for i in range(static_Y)]

    hidden_partial = In_local @ Win_local
    hidden_local = jax.lax.psum_scatter(
        hidden_partial,
        axis_name="Y",
        scatter_dimension=1, # Scatter along the F dimension (dimension 1)
        tiled=True,
    )

    def contribution_for_d_tile(d_owner):
        d0 = d_owner * D_Y
        Wout_tile = jax.lax.dynamic_slice(
            Wout_local,
            (0, d0),
            (F_Y, D_Y),
        )
        return (hidden_local @ Wout_tile).astype(Q33_RING_ACCUM_DTYPE)

    # Same no-return-hop ring schedule as Q3.2.
    tile_owner = (y - 1) % static_Y
    accum = contribution_for_d_tile(tile_owner)

    for _ in range(static_Y - 1):
        accum = jax.lax.ppermute(accum, axis_name="Y", perm=perm)
        tile_owner = (tile_owner - 1) % static_Y
        accum = (accum + contribution_for_d_tile(tile_owner)).astype(Q33_RING_ACCUM_DTYPE)

    return accum.astype(In_local.dtype)


def test_transformer_block_impls():
    with jax.set_mesh(mesh_2d):
        plain_jit = jax.jit(transformer_block_reference, out_shardings=out5_sharding)
        baseline_jit = jax.jit(transformer_block_baseline, out_shardings=out5_sharding)
        tiled_jit = jax.jit(transformer_block_tiled, out_shardings=out5_sharding)

        y_plain = plain_jit(In5, Win5, Wout5).block_until_ready()
        y_baseline = baseline_jit(In5, Win5, Wout5).block_until_ready()
        y_tiled = tiled_jit(In5, Win5, Wout5).block_until_ready()

        np.testing.assert_allclose(np.array(y_baseline), np.array(y_plain), rtol=Q33_RTOL, atol=Q33_ATOL)
        np.testing.assert_allclose(np.array(y_tiled), np.array(y_baseline), rtol=Q33_RTOL, atol=Q33_ATOL)

        hlo_plain = plain_jit.lower(In5, Win5, Wout5).compile().as_text()
        hlo_baseline = baseline_jit.lower(In5, Win5, Wout5).compile().as_text()
        hlo_tiled = tiled_jit.lower(In5, Win5, Wout5).compile().as_text()

        print("plain jit collectives:")
        for line in hlo_plain.splitlines():
            if any(x in line.lower() for x in ["all-reduce", "collective-permute", "all-gather", "all-to-all"]):
                print(" ", line.strip())

        print("baseline collectives:")
        for line in hlo_baseline.splitlines():
            if any(x in line.lower() for x in ["all-reduce", "collective-permute", "all-gather", "all-to-all"]):
                print(" ", line.strip())

        print("tiled collectives:")
        for line in hlo_tiled.splitlines():
            if any(x in line.lower() for x in ["all-reduce", "collective-permute", "all-gather", "all-to-all"]):
                print(" ", line.strip())


# This is intentionally commented because the TPU-sized defaults are large.
test_transformer_block_impls()


plain jit collectives:
  %collective-permute-start.2 = (bf16[4096,2048]{1,0:T(8,128)(2,1)S(1)}, bf16[4096,2048]{1,0:T(8,128)(2,1)S(1)}, u32[]{:S(2)}, u32[]{:S(2)}) collective-permute-start(%broadcast), channel_id=2, source_target_pairs={{0,3},{1,0},{2,1},{3,2},{4,7},{5,4},{6,5},{7,6}}, metadata={op_name="jit(transformer_block_reference)/dot_general" stack_frame_id=21}, backend_config={"barrier_config":{"barrier_type":"CUSTOM","id":"1"},"flag_configs":[],"scoped_memory_configs":[],"used_scoped_memory_configs":[]}
  %collective-permute-start.4 = (bf16[4096,2048]{1,0:T(8,128)(2,1)S(1)}, bf16[4096,2048]{1,0:T(8,128)(2,1)S(1)}, u32[]{:S(2)}, u32[]{:S(2)}) collective-permute-start(%broadcast), channel_id=3, source_target_pairs={{0,1},{1,2},{2,3},{3,0},{4,5},{5,6},{6,7},{7,4}}, metadata={op_name="jit(transformer_block_reference)/dot_general" stack_frame_id=21}, backend_config={"barrier_config":{"barrier_type":"CUSTOM","id":"0"},"flag_configs":[],"scoped_memory_configs":[],"used_scoped_memory_

In [17]:
# Question 3.3 timing comparison

import time


def benchmark_ms(label, fn, *args, warmup=5, iters=20):
    for _ in range(warmup):
        jax.block_until_ready(fn(*args))

    start = time.perf_counter()
    for _ in range(iters):
        jax.block_until_ready(fn(*args))
    elapsed = time.perf_counter() - start

    ms = 1000 * elapsed / iters
    print(f"{label:36s} {ms:8.3f} ms/call")
    return ms


with jax.set_mesh(mesh_2d):
    q33_warmup = int(os.environ.get("SECTION10_Q33_WARMUP", "5"))
    q33_iters = int(os.environ.get("SECTION10_Q33_ITERS", "20" if Q33_ON_TPU else "10"))

    plain_jit_33 = jax.jit(transformer_block_reference, out_shardings=out5_sharding)
    baseline_jit_33 = jax.jit(transformer_block_baseline, out_shardings=out5_sharding)
    tiled_jit_33 = jax.jit(transformer_block_tiled, out_shardings=out5_sharding)

    print("Question 3.3 config:")
    print("  backend:", jax.default_backend())
    print("  mesh:", mesh_2d)
    print("  shapes:", {"In": In5.shape, "Win": Win5.shape, "Wout": Wout5.shape, "out": (B5, D5)})
    print("  matmul dtype:", Q33_MATMUL_DTYPE)
    print("  ring accumulator dtype:", Q33_RING_ACCUM_DTYPE)
    print("  tolerances:", {"rtol": Q33_RTOL, "atol": Q33_ATOL})
    print("  benchmark:", {"warmup": q33_warmup, "iters": q33_iters})

    y_plain_33 = plain_jit_33(In5, Win5, Wout5).block_until_ready()
    y_baseline_33 = baseline_jit_33(In5, Win5, Wout5).block_until_ready()
    y_tiled_33 = tiled_jit_33(In5, Win5, Wout5).block_until_ready()

    np.testing.assert_allclose(np.array(y_baseline_33), np.array(y_plain_33), rtol=Q33_RTOL, atol=Q33_ATOL)
    np.testing.assert_allclose(np.array(y_tiled_33), np.array(y_baseline_33), rtol=Q33_RTOL, atol=Q33_ATOL)

    plain_ms = benchmark_ms(
        "3.3 plain jax.jit baseline",
        plain_jit_33,
        In5,
        Win5,
        Wout5,
        warmup=q33_warmup,
        iters=q33_iters,
    )
    explicit_baseline_ms = benchmark_ms(
        "3.3 shard_map baseline",
        baseline_jit_33,
        In5,
        Win5,
        Wout5,
        warmup=q33_warmup,
        iters=q33_iters,
    )
    tiled_ms = benchmark_ms(
        "3.3 hybrid ring bf16",
        tiled_jit_33,
        In5,
        Win5,
        Wout5,
        warmup=q33_warmup,
        iters=q33_iters,
    )

    print(f"hybrid vs plain jit speedup:       {plain_ms / tiled_ms:8.3f}x")
    print(f"hybrid vs shard_map baseline:     {explicit_baseline_ms / tiled_ms:8.3f}x")


Question 3.3 config:
  backend: tpu
  mesh: Mesh('X': 2, 'Y': 4, axis_types=(Auto, Auto))
  shapes: {'In': (8192, 16384), 'Win': (16384, 8192), 'Wout': (8192, 16384), 'out': (8192, 16384)}
  matmul dtype: <class 'jax.numpy.bfloat16'>
  ring accumulator dtype: <class 'jax.numpy.bfloat16'>
  tolerances: {'rtol': 0.05, 'atol': 512.0}
  benchmark: {'warmup': 5, 'iters': 20}
3.3 plain jax.jit baseline              6.574 ms/call
3.3 shard_map baseline                  6.944 ms/call
3.3 hybrid ring bf16                    6.212 ms/call
hybrid vs plain jit speedup:          1.058x
hybrid vs shard_map baseline:        1.118x


In [18]:
# Question 4: bidirectional ppermute collective matmuls

# Q4 asks us to take the explicit ppermute collectives and use both directions
# of the Y ring. This cell reuses the dimensions, dtypes, mesh, and tolerances
# from the Q3.1 and Q3.2 cells above.

import time

Q41_RTOL = float(os.environ.get("SECTION10_Q41_RTOL", str(Q31_RTOL)))
Q41_ATOL = float(os.environ.get("SECTION10_Q41_ATOL", str(Q31_ATOL)))
Q42_RTOL = float(os.environ.get("SECTION10_Q42_RTOL", str(Q32_RTOL)))
Q42_ATOL = float(os.environ.get("SECTION10_Q42_ATOL", str(Q32_ATOL)))

assert F3_tile % 2 == 0
assert D4_tile % 2 == 0


def benchmark_q4(label, fn, *args, warmup=5, iters=20):
    for _ in range(warmup):
        jax.block_until_ready(fn(*args))

    start = time.perf_counter()
    for _ in range(iters):
        jax.block_until_ready(fn(*args))
    elapsed = time.perf_counter() - start

    ms = 1000 * elapsed / iters
    print(f"{label:40s} {ms:8.3f} ms/call")
    return ms


@jax.shard_map(
    mesh=mesh_2d,
    in_specs=(jax.P("X", "Y"), jax.P("Y", None)),
    out_specs=jax.P("X", None),
    check_vma=False,
)
def allreduce_matmul_bidirectional(
    A_local: jnp.ndarray,
    W_local: jnp.ndarray,
) -> jnp.ndarray:
    # A_local: [B_X, D_Y]
    # W_local: [D_Y, F]
    # Return:  [B_X, F]
    B_X, D_Y = A_local.shape
    _, F = W_local.shape

    static_Y = mesh_2d.shape["Y"]
    forward_perm = [(i, (i + 1) % static_Y) for i in range(static_Y)]
    backward_perm = [(i, (i - 1) % static_Y) for i in range(static_Y)]

    out = jnp.zeros((B_X, F), dtype=Q31_REDUCE_DTYPE)
    half = F3_tile // 2

    for tile_idx in range(Q31_NUM_F_TILES):
        f0 = tile_idx * F3_tile
        W_tile = jax.lax.dynamic_slice(W_local, (0, f0), (D_Y, F3_tile))
        partial = (A_local @ W_tile).astype(Q31_REDUCE_DTYPE)

        forward_accum = jax.lax.dynamic_slice(partial, (0, 0), (B_X, half))
        backward_accum = jax.lax.dynamic_slice(partial, (0, half), (B_X, F3_tile - half))
        forward_send = forward_accum
        backward_send = backward_accum

        for _ in range(static_Y - 1):
            forward_send = jax.lax.ppermute(
                forward_send,
                axis_name="Y",
                perm=forward_perm,
            )
            backward_send = jax.lax.ppermute(
                backward_send,
                axis_name="Y",
                perm=backward_perm,
            )
            forward_accum = (forward_accum + forward_send).astype(Q31_REDUCE_DTYPE)
            backward_accum = (backward_accum + backward_send).astype(Q31_REDUCE_DTYPE)

        tile = jnp.concatenate([forward_accum, backward_accum], axis=1)
        out = jax.lax.dynamic_update_slice(out, tile, (0, f0))

    return out.astype(A_local.dtype)


@jax.shard_map(
    mesh=mesh_2d,
    in_specs=(jax.P("X", "Y"), jax.P("Y", None)),
    out_specs=jax.P("X", "Y"),
    check_vma=False,
)
def reducescatter_matmul_bidirectional(
    Tmp_local: jnp.ndarray,
    W2_local: jnp.ndarray,
) -> jnp.ndarray:
    # Tmp_local: [B_X, F_Y]
    # W2_local:  [F_Y, D]
    # Return:    [B_X, D_Y]
    y = jax.lax.axis_index("Y")
    B_X, F_Y = Tmp_local.shape

    static_Y = mesh_2d.shape["Y"]
    forward_perm = [(i, (i + 1) % static_Y) for i in range(static_Y)]
    backward_perm = [(i, (i - 1) % static_Y) for i in range(static_Y)]
    half = D4_tile // 2

    def contribution_for_tile(tile_owner, col_offset, width):
        d0 = tile_owner * D4_tile + col_offset
        W_tile = jax.lax.dynamic_slice(
            W2_local,
            (0, d0),
            (F_Y, width),
        )
        return (Tmp_local @ W_tile).astype(Q32_RING_ACCUM_DTYPE)

    # First half of each output D_Y tile travels forward. The second half
    # travels backward. Each half starts one hop away from its final owner so
    # after static_Y - 1 permutes it lands on the owning Y device.
    forward_owner = (y - 1) % static_Y
    backward_owner = (y + 1) % static_Y
    forward_accum = contribution_for_tile(forward_owner, 0, half)
    backward_accum = contribution_for_tile(backward_owner, half, D4_tile - half)

    for _ in range(static_Y - 1):
        forward_accum = jax.lax.ppermute(
            forward_accum,
            axis_name="Y",
            perm=forward_perm,
        )
        forward_owner = (forward_owner - 1) % static_Y
        forward_accum = (
            forward_accum + contribution_for_tile(forward_owner, 0, half)
        ).astype(Q32_RING_ACCUM_DTYPE)

        backward_accum = jax.lax.ppermute(
            backward_accum,
            axis_name="Y",
            perm=backward_perm,
        )
        backward_owner = (backward_owner + 1) % static_Y
        backward_accum = (
            backward_accum + contribution_for_tile(backward_owner, half, D4_tile - half)
        ).astype(Q32_RING_ACCUM_DTYPE)

    return jnp.concatenate([forward_accum, backward_accum], axis=1).astype(Tmp_local.dtype)


def run_q4_bidirectional_benchmarks():
    q4_warmup = int(os.environ.get("SECTION10_Q4_WARMUP", "5"))
    q4_iters = int(os.environ.get("SECTION10_Q4_ITERS", "20" if jax.default_backend() == "tpu" else "10"))

    with jax.set_mesh(mesh_2d):
        psum_ar_jit = jax.jit(allreduce_matmul_baseline, out_shardings=out3_sharding)
        tiled_ar_jit = jax.jit(allreduce_matmul_tiled, out_shardings=out3_sharding)
        bidi_ar_jit = jax.jit(allreduce_matmul_bidirectional, out_shardings=out3_sharding)

        baseline_rs_jit = jax.jit(reducescatter_matmul_baseline, out_shardings=out4_sharding)
        unidir_rs_jit = jax.jit(reducescatter_matmul_tiled, out_shardings=out4_sharding)
        bidi_rs_jit = jax.jit(reducescatter_matmul_bidirectional, out_shardings=out4_sharding)

        print("Question 4 config:")
        print("  backend:", jax.default_backend())
        print("  mesh:", mesh_2d)
        print("  benchmark:", {"warmup": q4_warmup, "iters": q4_iters})
        print("  Q4.1 shapes:", {"A": A3.shape, "W": W3.shape, "out": (B3, F3)})
        print("  Q4.1 dtypes:", {"matmul": Q31_MATMUL_DTYPE, "reduce": Q31_REDUCE_DTYPE})
        print("  Q4.1 tolerances:", {"rtol": Q41_RTOL, "atol": Q41_ATOL})
        print("  Q4.2 shapes:", {"Tmp": Tmp4.shape, "W2": W4.shape, "out": (B4, D4)})
        print("  Q4.2 dtypes:", {"matmul": Q32_MATMUL_DTYPE, "ring_accum": Q32_RING_ACCUM_DTYPE})
        print("  Q4.2 tolerances:", {"rtol": Q42_RTOL, "atol": Q42_ATOL})

        y_ar_psum = psum_ar_jit(A3, W3).block_until_ready()
        y_ar_tiled = tiled_ar_jit(A3, W3).block_until_ready()
        y_ar_bidi = bidi_ar_jit(A3, W3).block_until_ready()
        np.testing.assert_allclose(np.array(y_ar_tiled), np.array(y_ar_psum), rtol=Q41_RTOL, atol=Q41_ATOL)
        np.testing.assert_allclose(np.array(y_ar_bidi), np.array(y_ar_psum), rtol=Q41_RTOL, atol=Q41_ATOL)

        y_rs_baseline = baseline_rs_jit(Tmp4, W4).block_until_ready()
        y_rs_unidir = unidir_rs_jit(Tmp4, W4).block_until_ready()
        y_rs_bidi = bidi_rs_jit(Tmp4, W4).block_until_ready()
        np.testing.assert_allclose(np.array(y_rs_unidir), np.array(y_rs_baseline), rtol=Q42_RTOL, atol=Q42_ATOL)
        np.testing.assert_allclose(np.array(y_rs_bidi), np.array(y_rs_baseline), rtol=Q42_RTOL, atol=Q42_ATOL)

        print()
        print("Question 4.1: AllReduce matmul")
        q41_psum_ms = benchmark_q4(
            "3.1 shard_map psum",
            psum_ar_jit,
            A3,
            W3,
            warmup=q4_warmup,
            iters=q4_iters,
        )
        q41_tiled_ms = benchmark_q4(
            "3.1 tiled psum bf16",
            tiled_ar_jit,
            A3,
            W3,
            warmup=q4_warmup,
            iters=q4_iters,
        )
        q41_bidi_ms = benchmark_q4(
            "4.1 bidirectional ppermute",
            bidi_ar_jit,
            A3,
            W3,
            warmup=q4_warmup,
            iters=q4_iters,
        )
        print(f"bidirectional vs psum baseline: {q41_psum_ms / q41_bidi_ms:8.3f}x")
        print(f"bidirectional vs tiled psum:    {q41_tiled_ms / q41_bidi_ms:8.3f}x")

        print()
        print("Question 4.2: ReduceScatter matmul")
        q42_baseline_ms = benchmark_q4(
            "3.2 baseline psum",
            baseline_rs_jit,
            Tmp4,
            W4,
            warmup=q4_warmup,
            iters=q4_iters,
        )
        q42_unidir_ms = benchmark_q4(
            "3.2 unidirectional ppermute",
            unidir_rs_jit,
            Tmp4,
            W4,
            warmup=q4_warmup,
            iters=q4_iters,
        )
        q42_bidi_ms = benchmark_q4(
            "4.2 bidirectional ppermute",
            bidi_rs_jit,
            Tmp4,
            W4,
            warmup=q4_warmup,
            iters=q4_iters,
        )
        print(f"bidirectional vs psum baseline: {q42_baseline_ms / q42_bidi_ms:8.3f}x")
        print(f"bidirectional vs unidirectional:{q42_unidir_ms / q42_bidi_ms:8.3f}x")


run_q4_bidirectional_benchmarks()


Question 4 config:
  backend: tpu
  mesh: Mesh('X': 2, 'Y': 4, axis_types=(Auto, Auto))
  benchmark: {'warmup': 5, 'iters': 20}
  Q4.1 shapes: {'A': (8192, 16384), 'W': (16384, 8192), 'out': (8192, 8192)}
  Q4.1 dtypes: {'matmul': <class 'jax.numpy.bfloat16'>, 'reduce': <class 'jax.numpy.bfloat16'>}
  Q4.1 tolerances: {'rtol': 0.05, 'atol': 32.0}
  Q4.2 shapes: {'Tmp': (8192, 16384), 'W2': (16384, 8192), 'out': (8192, 8192)}
  Q4.2 dtypes: {'matmul': <class 'jax.numpy.bfloat16'>, 'ring_accum': <class 'jax.numpy.bfloat16'>}
  Q4.2 tolerances: {'rtol': 0.05, 'atol': 8.0}

Question 4.1: AllReduce matmul
3.1 shard_map psum                          3.004 ms/call
3.1 tiled psum bf16                         3.178 ms/call
4.1 bidirectional ppermute                  5.633 ms/call
bidirectional vs psum baseline:    0.533x
bidirectional vs tiled psum:       0.564x

Question 4.2: ReduceScatter matmul
3.2 baseline psum                           3.029 ms/call
3.2 unidirectional ppermute             